In [1]:
import pandas as pd
import ast

# -----------------------------
# 1) Load Dataset
# -----------------------------
df = pd.read_csv("customer_data_collection.csv")

print("Initial shape:", df.shape)
print(df.head())

Initial shape: (10000, 11)
  Customer_ID  Age  Gender   Location                 Browsing_History  \
0       C1000   28  Female    Chennai             ['Books', 'Fashion']   
1       C1001   27    Male      Delhi  ['Books', 'Fitness', 'Fashion']   
2       C1002   34   Other    Chennai                  ['Electronics']   
3       C1003   23    Male  Bangalore                   ['Home Decor']   
4       C1004   24   Other    Kolkata        ['Fashion', 'Home Decor']   

                               Purchase_History    Customer_Segment  \
0                        ['Biography', 'Jeans']         New Visitor   
1  ['Biography', 'Resistance Bands', 'T-shirt']  Occasional Shopper   
2                                ['Smartphone']  Occasional Shopper   
3                                  ['Wall Art']      Frequent Buyer   
4                             ['Shoes', 'Lamp']      Frequent Buyer   

   Avg_Order_Value Holiday  Season  Unnamed: 10  
0          4806.99      No  Winter          NaN  
1

In [2]:
# -----------------------------
# 2) Check for NULL values
# -----------------------------
print("\n----- NULL VALUES -----")
print(df.isnull().sum())

# Show rows having NULL in important columns
print("\nRows with null in Customer_ID or Purchase_History:")
print(df[df['Customer_ID'].isnull() | df['Purchase_History'].isnull()])



----- NULL VALUES -----
Customer_ID             0
Age                     0
Gender                  0
Location                0
Browsing_History        0
Purchase_History        0
Customer_Segment        0
Avg_Order_Value         0
Holiday                 0
Season                  0
Unnamed: 10         10000
dtype: int64

Rows with null in Customer_ID or Purchase_History:
Empty DataFrame
Columns: [Customer_ID, Age, Gender, Location, Browsing_History, Purchase_History, Customer_Segment, Avg_Order_Value, Holiday, Season, Unnamed: 10]
Index: []


In [3]:
# -----------------------------
# 3) Check for duplicates
# -----------------------------
print("\n----- DUPLICATES -----")
print("Total duplicates:", df.duplicated().sum())

# Display duplicate rows (if any)
duplicates = df[df.duplicated()]
print(duplicates)


----- DUPLICATES -----
Total duplicates: 0
Empty DataFrame
Columns: [Customer_ID, Age, Gender, Location, Browsing_History, Purchase_History, Customer_Segment, Avg_Order_Value, Holiday, Season, Unnamed: 10]
Index: []


In [4]:
# -----------------------------
# 4) Data Cleaning
# -----------------------------
df_clean = df.copy()

# Remove duplicates
df_clean.drop_duplicates(inplace=True)

# Remove rows with null in Customer_ID or Purchase_History
df_clean.dropna(subset=['Customer_ID', 'Purchase_History'], inplace=True)

df_clean['Customer_ID'] = df_clean['Customer_ID'].astype(str)

print("\nAfter cleaning shape:", df_clean.shape)


After cleaning shape: (10000, 11)


In [5]:
# -----------------------------
# 5) Convert Purchase_History from string to list
# -----------------------------
def str_to_list(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

df_clean['Purchase_History'] = df_clean['Purchase_History'].apply(str_to_list)

In [6]:
# -----------------------------
# 6) Explode purchase history into rows
# -----------------------------
df_exploded = df_clean.explode('Purchase_History')

# Rename for clarity
df_exploded = df_exploded.rename(columns={'Purchase_History': 'product'})

# Clean product strings
df_exploded['product'] = df_exploded['product'].astype(str).str.strip()

# Remove empty strings
df_exploded = df_exploded[df_exploded['product'] != ""]

print("\nExploded shape:", df_exploded.shape)
print(df_exploded.head())


Exploded shape: (19967, 11)
  Customer_ID  Age  Gender Location                 Browsing_History  \
0       C1000   28  Female  Chennai             ['Books', 'Fashion']   
0       C1000   28  Female  Chennai             ['Books', 'Fashion']   
1       C1001   27    Male    Delhi  ['Books', 'Fitness', 'Fashion']   
1       C1001   27    Male    Delhi  ['Books', 'Fitness', 'Fashion']   
1       C1001   27    Male    Delhi  ['Books', 'Fitness', 'Fashion']   

            product    Customer_Segment  Avg_Order_Value Holiday  Season  \
0         Biography         New Visitor          4806.99      No  Winter   
0             Jeans         New Visitor          4806.99      No  Winter   
1         Biography  Occasional Shopper           795.03     Yes  Autumn   
1  Resistance Bands  Occasional Shopper           795.03     Yes  Autumn   
1           T-shirt  Occasional Shopper           795.03     Yes  Autumn   

   Unnamed: 10  
0          NaN  
0          NaN  
1          NaN  
1          Na

In [7]:
# -----------------------------
# 7) Check duplicates after explode
# -----------------------------
print("\nDuplicates after explode:", df_exploded.duplicated(subset=['Customer_ID', 'product']).sum())


Duplicates after explode: 0


In [8]:
# -----------------------------
# 8) Build interaction score
# -----------------------------
interaction_matrix = df_exploded.groupby(
    ['Customer_ID', 'product']
).size().unstack(fill_value=0)

print(interaction_matrix.head())


product      Biography  Comics  Curtains  Cushions  Dumbbells  Fiction  \
Customer_ID                                                              
C1000                1       0         0         0          0        0   
C10000               0       0         0         0          0        0   
C10001               0       0         0         0          0        0   
C10002               0       0         0         0          0        0   
C10003               0       0         0         1          0        0   

product      Foundation  Headphones  Jacket  Jeans  ...  Non-fiction  Perfume  \
Customer_ID                                         ...                         
C1000                 0           0       0      1  ...            0        0   
C10000                0           0       0      0  ...            0        1   
C10001                0           0       0      0  ...            0        0   
C10002                0           0       0      0  ...            0        

In [9]:
# -----------------------------
# 10) Save Cleaned Data & Matrix
# -----------------------------
df_exploded.to_csv("cleaned_dataset.csv", index=False)
interaction_matrix.to_csv("user_item_matrix.csv")

print("\nSaved cleaned dataset and user-item matrix.")


Saved cleaned dataset and user-item matrix.
